In [125]:
import pandas as pd

In [126]:
df=pd.read_csv("IMDB_Dataset.csv")

In [127]:
df.shape

(50000, 2)

In [128]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [129]:
df.drop_duplicates(inplace=True)

In [130]:
df.shape

(49582, 2)

In [131]:
df["review"]=df["review"].str.lower()

In [132]:
import re

In [133]:
def remove_urls(text):
    return re.sub(r"http\S+","",text)
df["review"]=df["review"].apply(remove_urls)

In [134]:
def remove_punctuations(text):
    return re.sub(r"[^A-Za-z0-9\s]","",text)
df["review"]=df["review"].apply(remove_punctuations)

In [135]:
def remove_html(text):
    return re.sub(r"<,*?>","",text)

In [136]:
import nltk

In [137]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


In [138]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")

    return text
df["review"]=df["review"].apply(remove_stopwords)

In [139]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [140]:
from nltk.stem import PorterStemmer

In [141]:
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(stemming)

In [142]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [143]:
y=df["sentiment"]

In [144]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [161]:
tf=TfidfVectorizer(max_features=5000)

In [162]:
x=tf.fit_transform(df["review"])

In [163]:
from sklearn.model_selection import train_test_split

In [164]:
x_test,x_train,y_test,y_train=train_test_split(x,y, test_size=0.2,random_state=42)

In [165]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [166]:
x_train = x_train.toarray()
x_test = x_test.toarray()

In [168]:
train_set = TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [169]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [170]:
import torch.nn as nn
import torch.optim as optim

In [171]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        out = self.fc(out[:, -1, :])
        return out

In [173]:
input_size = x_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)
        
        outputs = model(Xb)

        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")